# Hyperparameter Optimization (Optuna) — 3 Kandidat Teratas

`CBQD - 4Fold CV.ipynb` menemukan peringkat `09_noise_robust` > `05_convnext_tiny` >
`08_multitask` TIDAK stabil per-fold (setiap model pernah jadi juara di minimal 1 fold,
rentang skornya saling tumpang tindih). Salah satu kemungkinan penyebab: ketiganya
memakai SATU resep hyperparameter yang sama (dipilih untuk kelayakan waktu 10 model
sekaligus di `CBQD - Training.ipynb`), belum tentu optimal untuk masing-masing arsitektur.
Notebook ini memberi tiap model kesempatan yang adil lewat Optuna, baru dibandingkan lagi.

**Parameter yang dioptimasi (rancangan disetujui sebelum notebook ini ditulis):**

| Kategori | Parameter | Rentang | Model |
|---|---|---|---|
| Bersama | `lr_phase1` | 1e-4 – 3e-3 (log) | Semua |
| Bersama | `lr_phase2` | 1e-5 – 1e-3 (log) | Semua |
| Bersama | `weight_decay` | 1e-4 – 0,1 (log) | Semua |
| Bersama | `batch_size` | {16, 32, 64} | Semua |
| Bersama | `scheduler_factor` | 0,3 – 0,7 | Semua |
| Bersama | `scheduler_patience` | {2, 3, 4, 6} | Semua |
| Khusus | `label_smoothing` | 0,0 – 0,2 | 09_noise_robust |
| Khusus | `mislabel_weight` | 0,1 – 0,9 | 09_noise_robust |
| Khusus | `mistake_threshold` | 0,3 – 0,7 | 09_noise_robust |
| Khusus | `type_loss_weight` | 0,2 – 5,0 (log, **baru** — resep asli implisit 1,0) | 08_multitask |

**Protokol pencarian:**
- Search di SATU split tetap (val=`cv_fold==0`, fit=`cv_fold∈{1,2,3}`) — sama seperti
  `CBQD - Training.ipynb` — supaya tiap trial murah, bukan 4-fold CV per trial (4x lipat
  biaya, terlalu mahal untuk fase pencarian).
- Sampler TPE, pruner `MedianPruner` (dilaporkan tiap epoch fase-2) — trial yang jelas
  kalah dihentikan dini.
- **30 trial/model** (disepakati) — dengan `EPOCHS_PHASE2` fase pencarian dikecilkan ke
  30 (dari 45 di resep asli) supaya total runtime (30 trial x 3 model, dengan pruning)
  tetap masuk akal untuk satu sesi Kaggle. Trial yang benar-benar bagus tetap dapat
  early-stop patience penuh (10), cuma plafon maksimalnya yang dipangkas.
- Setelah `study.optimize()` selesai per model, hyperparameter TERBAIK dipakai untuk SATU
  kali retrain final dengan plafon epoch penuh (45) -- ini yang dilaporkan sebagai angka
  final (val + test), bukan angka dari trial pencarian yang sengaja dipangkas.
- Studi Optuna disimpan ke SQLite (`results/optuna_<model>.db`) -- kalau kernel berhenti
  di tengah jalan, trial yang sudah selesai tidak hilang & studi bisa dilanjutkan.
- Checkpoint model tuned TIDAK didorong ke DVC di notebook ini (sama seperti
  `CBQD - 4Fold CV.ipynb`) -- nilainya ada di angka val/test yang dilaporkan.

**PENTING — DRY_RUN:** `DRY_RUN=True` -> 3 trial/model + epoch kecil, validasi seluruh
pipeline Optuna (suggest, prune, report, retrain final) jalan tanpa error dulu.

## Section 1 — Environment & Data Provenance Setup

In [ ]:
# Sub-Step 1.1
# Tujuan: Clone repo + install optuna (tanpa menyentuh torch/torchvision)

!pip install -q optuna

import os
from pathlib import Path

GIT_REPO_URL = "https://github.com/Ardiyanto24/coffee-bean-quality-detection.git"
PROJECT_DIR = "/kaggle/working/coffee-bean-quality-detection"
if not os.path.exists(PROJECT_DIR):
    os.system(f"git clone {GIT_REPO_URL} {PROJECT_DIR}")
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Sub-Step 1.2
# Tujuan: Konfigurasi kredensial R2 (private dataset jika ada, fallback ke Kaggle Secrets)

import json

_matches = list(Path("/kaggle/input").rglob("r2_credentials.json")) if os.path.exists("/kaggle/input") else []
cred_path = _matches[0] if _matches else None

if cred_path is not None:
    creds = json.loads(cred_path.read_text())
    os.environ["AWS_ACCESS_KEY_ID"] = creds["R2_ACCESS_KEY_ID"]
    os.environ["AWS_SECRET_ACCESS_KEY"] = creds["R2_SECRET_ACCESS_KEY"]
    print("Kredensial R2 dimuat dari private Kaggle Dataset (nilai tidak di-print).")
else:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["AWS_ACCESS_KEY_ID"] = secrets.get_secret("R2_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = secrets.get_secret("R2_SECRET_ACCESS_KEY")
    print("Kredensial R2 dimuat dari Kaggle Secrets (nilai tidak di-print).")

In [ ]:
# Sub-Step 1.3
# Tujuan: Tarik dataset + manifest dari R2 (dvc pull)

!pip install -q "dvc[s3]"
!dvc pull -v
print("dataset/ ada:", Path("dataset").exists())
print("dataset_preprocessed/ ada:", Path("dataset_preprocessed").exists())
print("manifest_preprocessed.csv ada:", Path("metadata/manifest_preprocessed.csv").exists())

## Section 2 — Konfigurasi Eksperimen

In [ ]:
# Sub-Step 2.1
# Tujuan: Flag DRY_RUN + turunan epoch/trial; hyperparameter TETAP (bukan yang dioptimasi)

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DRY_RUN = False  # <-- dry-run (3 trial/model, epoch kecil, pruning genuinely teruji lewat
                 # startup kecil) sudah lolos bersih; ini full run 30 trial/model

if DRY_RUN:
    EPOCHS_PHASE1 = 1
    EPOCHS_PHASE2_SEARCH = 3
    EPOCHS_PHASE2_FINAL = 4
    EARLY_STOP_PATIENCE = 2
    N_TRIALS = 3
    PRUNER_STARTUP_TRIALS = 1  # kecil di dry-run -- supaya jalur TrialPruned() genuinely
    PRUNER_WARMUP_STEPS = 1    # dites (bukan cuma trial.report() tanpa pernah prune)
else:
    EPOCHS_PHASE1 = 5
    EPOCHS_PHASE2_SEARCH = 30   # dipangkas dari 45 (resep asli) -- trial pencarian cukup
                                # sinyal relatif, bukan konvergensi penuh
    EPOCHS_PHASE2_FINAL = 45    # retrain final (hyperparameter terbaik) pakai plafon penuh
    EARLY_STOP_PATIENCE = 10
    N_TRIALS = 30
    PRUNER_STARTUP_TRIALS = 5
    PRUNER_WARMUP_STEPS = 5

IMG_SIZE = 224
CLASS_NAMES = ["defect", "longberry", "peaberry", "premium"]
LABEL_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
TYPE_TO_FLAT = {0: LABEL_TO_IDX["premium"], 1: LABEL_TO_IDX["peaberry"], 2: LABEL_TO_IDX["longberry"]}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DRY_RUN={DRY_RUN} | device={device} | n_trials={N_TRIALS} | epochs phase1/phase2_search/phase2_final={EPOCHS_PHASE1}/{EPOCHS_PHASE2_SEARCH}/{EPOCHS_PHASE2_FINAL}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print("GPU:", gpu_name)
    if "P100" in gpu_name:
        raise RuntimeError(
            f"GPU allocated is {gpu_name}, incompatible with the preinstalled PyTorch "
            "build (no Pascal/sm_60 kernels). Re-push the kernel with "
            "kernel-metadata.json machine_shape=NvidiaTeslaT4 to force a T4 allocation."
        )

## Section 3 — Data: Manifest & Dataset Classes

Split TETAP (sama seperti `CBQD - Training.ipynb`, BUKAN rotasi 4-fold seperti
`CBQD - 4Fold CV.ipynb`) -- HPO butuh 1 val konsisten di seluruh trial agar skor antar
trial sebanding. `batch_size` termasuk yang dioptimasi, jadi DataLoader dibangun per
trial di dalam objective(), bukan sekali di sini.

In [ ]:
# Sub-Step 3.1
# Tujuan: Load manifest, fit/val (tetap) & test (held-out)

import pandas as pd

manifest = pd.read_csv("metadata/manifest_preprocessed.csv")
PREP_DIR = Path("dataset_preprocessed")
RAW_DIR = Path("dataset")

train_pool = manifest[manifest["split"] == "train"].reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)

fit_df = train_pool[train_pool["cv_fold"].isin([1, 2, 3])].reset_index(drop=True)
val_df = train_pool[train_pool["cv_fold"] == 0].reset_index(drop=True)

print(f"fit={len(fit_df)}  val={len(val_df)}  test={len(test_df)}")

In [ ]:
# Sub-Step 3.2
# Tujuan: Dataset & transform (augmentasi rentang untuk train, resize+normalize untuk val/test) -- identik CBQD - Training.ipynb

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=180),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class BeanDataset(Dataset):
    """Dataset flat 4-class. `weights` opsional dipakai 09_noise_robust (default 1.0)."""

    def __init__(self, df, root_dir, transform, weights=None):
        self.paths = [root_dir / p for p in df["image_path"]]
        self.labels = [LABEL_TO_IDX[l] for l in df["label"]]
        self.transform = transform
        self.weights = weights if weights is not None else [1.0] * len(df)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.labels[idx], self.weights[idx]


class MultiTaskDataset(Dataset):
    """Untuk 08_multitask: (image, damage_label 0/1, type_label 0-2 atau -1 untuk defect)."""

    TYPE_MAP = {"premium": 0, "peaberry": 1, "longberry": 2}

    def __init__(self, df, root_dir, transform):
        self.paths = [root_dir / p for p in df["image_path"]]
        self.damage_labels = [1 if l == "defect" else 0 for l in df["label"]]
        self.type_labels = [self.TYPE_MAP.get(l, -1) for l in df["label"]]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.damage_labels[idx], self.type_labels[idx]


def make_loaders(dataset_cls, fit_kwargs, val_kwargs, batch_size):
    fit_loader = DataLoader(dataset_cls(fit_df, PREP_DIR, train_transform, **fit_kwargs),
                             batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(dataset_cls(val_df, PREP_DIR, eval_transform, **val_kwargs),
                             batch_size=batch_size, shuffle=False, num_workers=2)
    return fit_loader, val_loader


test_loader_cache = {}


def get_test_loader(batch_size):
    """test_loader tidak butuh augmentasi & selalu sama isinya -- cache per batch_size
    supaya tidak membangun ulang tiap kali dipanggil (dipanggil di tiap trial)."""
    if batch_size not in test_loader_cache:
        test_loader_cache[batch_size] = DataLoader(
            BeanDataset(test_df, PREP_DIR, eval_transform), batch_size=batch_size, shuffle=False, num_workers=2
        )
    return test_loader_cache[batch_size]

## Section 4 — Fungsi Utilitas Bersama (identik `CBQD - Training.ipynb`)

In [ ]:
# Sub-Step 4.1
# Tujuan: evaluate() & evaluate_combined(): macro-F1, accuracy, per-class report

import torch.nn as nn
from sklearn.metrics import f1_score, accuracy_score, classification_report


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels, _ in loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
    return {"macro_f1": macro_f1, "accuracy": acc, "report": report}


@torch.no_grad()
def evaluate_combined(predict_fn, loader, device):
    all_preds, all_labels = [], []
    for images, labels, _ in loader:
        images = images.to(device)
        preds = predict_fn(images)
        all_preds.extend(list(preds))
        all_labels.extend(labels.tolist())
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
    return {"macro_f1": macro_f1, "accuracy": acc, "report": report}

In [ ]:
# Sub-Step 4.2
# Tujuan: build_model() factory (convnext_tiny, efficientnet_b0) & EfficientNetMultiTask

from torchvision import models as tv_models


def build_model(arch, num_classes):
    if arch == "efficientnet_b0":
        m = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_f = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_f, num_classes)
        head = m.classifier[-1]
    elif arch == "convnext_tiny":
        m = tv_models.convnext_tiny(weights=tv_models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        in_f = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_f, num_classes)
        head = m.classifier[-1]
    else:
        raise ValueError(f"Arsitektur tidak dikenal: {arch}")
    return m, head


class EfficientNetMultiTask(nn.Module):
    """Untuk 08_multitask: satu backbone EfficientNet-B0 + 2 head terpisah."""

    def __init__(self):
        super().__init__()
        backbone = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_f = backbone.classifier[-1].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.head_damage = nn.Linear(in_f, 2)
        self.head_type = nn.Linear(in_f, 3)

    def forward(self, x):
        feats = self.backbone(x)
        return self.head_damage(feats), self.head_type(feats)


def set_backbone_frozen(model, head_module, frozen: bool):
    head_param_ids = set(id(p) for p in head_module.parameters())
    for p in model.parameters():
        p.requires_grad = (id(p) in head_param_ids) or (not frozen)

In [ ]:
# Sub-Step 4.3
# Tujuan: handcrafted_features() -- khusus skor mistakenness 09_noise_robust (identik Model 01)

import cv2


def handcrafted_features(path):
    bgr = cv2.imread(str(path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    gray_f = gray.astype(np.float32)
    thresh = gray_f.mean() - 0.6 * gray_f.std()
    mask = gray_f < thresh
    h, w = gray.shape
    if mask.sum() > 0:
        ys, xs = np.where(mask)
        area_frac = mask.sum() / (h * w)
        bbox_h, bbox_w = float(ys.max() - ys.min()), float(xs.max() - xs.min())
        bbox_ratio = max(bbox_h, bbox_w) / max(min(bbox_h, bbox_w), 1e-6)
        cy, cx = ys.mean(), xs.mean()
        center_offset = float(np.hypot(cy - h / 2, cx - w / 2) / (h / 2))
    else:
        area_frac = bbox_ratio = center_offset = np.nan

    edges = cv2.Canny(gray, 100, 200)
    return {
        "mean_r": rgb[:, :, 0].mean(), "mean_g": rgb[:, :, 1].mean(), "mean_b": rgb[:, :, 2].mean(),
        "std_r": rgb[:, :, 0].std(), "std_g": rgb[:, :, 1].std(), "std_b": rgb[:, :, 2].std(),
        "edge_density": edges.mean() / 255, "variance": float(np.var(gray)),
        "area_frac": area_frac, "bbox_ratio": bbox_ratio, "center_offset": center_offset,
    }


FEATURE_COLS = ["mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b",
                "edge_density", "variance", "area_frac", "bbox_ratio", "center_offset"]


def build_feature_matrix(df):
    rows = [handcrafted_features(RAW_DIR / p) for p in df["orig_path"]]
    X = pd.DataFrame(rows)[FEATURE_COLS].fillna(0.0).values
    y = np.array([LABEL_TO_IDX[l] for l in df["label"]])
    return X, y

In [ ]:
# Sub-Step 4.4
# Tujuan: Precompute mistake_score untuk 09_noise_robust SEKALI (tidak bergantung hyperparameter trial, cuma threshold-nya)

from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier

X_fit_m9, y_fit_m9 = build_feature_matrix(fit_df)
_sgkf_noise = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
_rf_noise = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
_proba_oof = cross_val_predict(_rf_noise, X_fit_m9, y_fit_m9, cv=_sgkf_noise,
                                groups=fit_df["cluster_id"].values, method="predict_proba")
_true_proba = _proba_oof[np.arange(len(y_fit_m9)), y_fit_m9]
_max_proba = _proba_oof.max(axis=1)
MISTAKE_SCORE = _max_proba - _true_proba  # array kontinu, tetap sepanjang HPO -- cuma
                                            # mistake_threshold (trial) yang mengubah cutoff-nya
print(f"MISTAKE_SCORE dihitung sekali untuk {len(fit_df)} gambar fit -- dipakai ulang di semua trial 09_noise_robust")

## Section 5 — Fungsi Training dengan Dukungan Optuna Pruning

Sama seperti `train_one_model()`/`train_multitask()` di notebook sebelumnya, tapi
hyperparameter jadi ARGUMEN (bukan konstanta global) dan tiap epoch fase-2 melapor ke
`trial.report()` supaya `MedianPruner` bisa menghentikan trial yang jelas kalah lebih awal.

In [ ]:
# Sub-Step 5.1
# Tujuan: train_one_model_hpo(): 2-fase + early stopping + pruning report

import copy

import optuna


def train_one_model_hpo(model, head_module, fit_loader, val_loader, device, model_name,
                         lr_phase1, lr_phase2, weight_decay, scheduler_factor, scheduler_patience,
                         epochs_phase2, criterion=None, trial=None):
    model = model.to(device)
    criterion = criterion if criterion is not None else nn.CrossEntropyLoss(reduction="none")
    best_state, best_val_f1, patience_counter = None, -1.0, 0

    set_backbone_frozen(model, head_module, frozen=True)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr_phase1, weight_decay=weight_decay
    )
    for epoch in range(EPOCHS_PHASE1):
        _train_epoch(model, fit_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, device)
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1, best_state, patience_counter = val_metrics["macro_f1"], copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1

    set_backbone_frozen(model, head_module, frozen=False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_phase2, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=scheduler_factor, patience=scheduler_patience)
    patience_counter = 0
    for epoch in range(epochs_phase2):
        _train_epoch(model, fit_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, device)
        scheduler.step(val_metrics["macro_f1"])
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1, best_state, patience_counter = val_metrics["macro_f1"], copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
        if trial is not None:
            trial.report(val_metrics["macro_f1"], epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
        if patience_counter >= EARLY_STOP_PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val_f1


def _train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    for images, labels, weights in loader:
        images, labels, weights = images.to(device), labels.to(device), weights.to(device).float()
        optimizer.zero_grad()
        outputs = model(images)
        per_sample_loss = criterion(outputs, labels)
        loss = (per_sample_loss * weights).mean() if per_sample_loss.dim() > 0 else per_sample_loss
        loss.backward()
        optimizer.step()

In [ ]:
# Sub-Step 5.2
# Tujuan: train_multitask_hpo(): loss gabungan berbobot (type_loss_weight) + pruning report

@torch.no_grad()
def _mt_val_f1(model, loader, device):
    model.eval()
    preds, labels_flat = [], []
    for images, damage_labels, type_labels in loader:
        images = images.to(device)
        out_damage, out_type = model(images)
        damage_pred = out_damage.argmax(dim=1).cpu().numpy()
        type_pred = out_type.argmax(dim=1).cpu().numpy()
        combined = np.where(damage_pred == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT[t] for t in type_pred])
        preds.extend(list(combined))
        dmg_np, typ_np = damage_labels.numpy(), type_labels.numpy()
        true_flat = np.where(dmg_np == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT.get(t, -1) for t in typ_np])
        labels_flat.extend(true_flat.tolist())
    return f1_score(labels_flat, preds, average="macro", zero_division=0)


def train_multitask_hpo(model, fit_loader, val_loader, device, model_name,
                         lr_phase1, lr_phase2, weight_decay, scheduler_factor, scheduler_patience,
                         epochs_phase2, type_loss_weight=1.0, trial=None):
    model = model.to(device)
    ce_damage, ce_type = nn.CrossEntropyLoss(), nn.CrossEntropyLoss()
    best_state, best_val_f1, patience_counter = None, -1.0, 0

    def run_epoch(optimizer):
        model.train()
        for images, damage_labels, type_labels in fit_loader:
            images = images.to(device); damage_labels = damage_labels.to(device); type_labels = type_labels.to(device)
            optimizer.zero_grad()
            out_damage, out_type = model(images)
            loss = ce_damage(out_damage, damage_labels)
            mask = type_labels >= 0
            if mask.any():
                loss = loss + type_loss_weight * ce_type(out_type[mask], type_labels[mask])
            loss.backward()
            optimizer.step()

    for p in model.backbone.parameters():
        p.requires_grad = False
    optimizer = torch.optim.AdamW(
        list(model.head_damage.parameters()) + list(model.head_type.parameters()),
        lr=lr_phase1, weight_decay=weight_decay,
    )
    for epoch in range(EPOCHS_PHASE1):
        run_epoch(optimizer)
        val_f1 = _mt_val_f1(model, val_loader, device)
        if val_f1 > best_val_f1:
            best_val_f1, best_state, patience_counter = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1

    for p in model.parameters():
        p.requires_grad = True
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_phase2, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=scheduler_factor, patience=scheduler_patience)
    patience_counter = 0
    for epoch in range(epochs_phase2):
        run_epoch(optimizer)
        val_f1 = _mt_val_f1(model, val_loader, device)
        scheduler.step(val_f1)
        if val_f1 > best_val_f1:
            best_val_f1, best_state, patience_counter = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
        if trial is not None:
            trial.report(val_f1, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
        if patience_counter >= EARLY_STOP_PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val_f1

## Section 6 — Studi Optuna per Model

Helper `suggest_shared_params()` dipakai ketiga objective supaya 6 parameter bersama
konsisten definisinya (rentang & tipe) di semua model.

In [ ]:
# Sub-Step 6.1
# Tujuan: Helper bersama: suggest_shared_params() & run_study()

def suggest_shared_params(trial):
    return {
        "lr_phase1": trial.suggest_float("lr_phase1", 1e-4, 3e-3, log=True),
        "lr_phase2": trial.suggest_float("lr_phase2", 1e-5, 1e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-4, 0.1, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64]),
        "scheduler_factor": trial.suggest_float("scheduler_factor", 0.3, 0.7),
        "scheduler_patience": trial.suggest_categorical("scheduler_patience", [2, 3, 4, 6]),
    }


def run_study(model_name, objective_fn, n_trials):
    Path("results").mkdir(exist_ok=True)
    storage = f"sqlite:///results/optuna_{model_name}.db"
    sampler = optuna.samplers.TPESampler(seed=SEED)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=PRUNER_STARTUP_TRIALS, n_warmup_steps=PRUNER_WARMUP_STEPS)
    study = optuna.create_study(study_name=model_name, storage=storage, load_if_exists=True,
                                 direction="maximize", sampler=sampler, pruner=pruner)
    study.optimize(objective_fn, n_trials=n_trials)
    trials_df = study.trials_dataframe()
    trials_df.to_csv(f"results/hpo_trials_{model_name}.csv", index=False)
    print(f"\n[{model_name}] {len(study.trials)} trial selesai. Terbaik: value={study.best_value:.4f}")
    print(f"[{model_name}] best_params: {study.best_params}")
    return study

In [ ]:
# Sub-Step 6.2
# Tujuan: Studi 1/3 -- 05_convnext_tiny

def objective_convnext(trial):
    p = suggest_shared_params(trial)
    fit_loader, val_loader = make_loaders(BeanDataset, {}, {}, p["batch_size"])
    model, head = build_model("convnext_tiny", num_classes=4)
    try:
        _, val_f1 = train_one_model_hpo(
            model, head, fit_loader, val_loader, device, f"05_convnext_tiny_trial{trial.number}",
            lr_phase1=p["lr_phase1"], lr_phase2=p["lr_phase2"], weight_decay=p["weight_decay"],
            scheduler_factor=p["scheduler_factor"], scheduler_patience=p["scheduler_patience"],
            epochs_phase2=EPOCHS_PHASE2_SEARCH, trial=trial,
        )
        return val_f1
    finally:
        del model
        torch.cuda.empty_cache()


study_convnext = run_study("05_convnext_tiny", objective_convnext, N_TRIALS)

In [ ]:
# Sub-Step 6.3
# Tujuan: Studi 2/3 -- 08_multitask

def objective_multitask(trial):
    p = suggest_shared_params(trial)
    type_loss_weight = trial.suggest_float("type_loss_weight", 0.2, 5.0, log=True)
    fit_loader, val_loader = make_loaders(MultiTaskDataset, {}, {}, p["batch_size"])
    model = EfficientNetMultiTask()
    try:
        _, val_f1 = train_multitask_hpo(
            model, fit_loader, val_loader, device, f"08_multitask_trial{trial.number}",
            lr_phase1=p["lr_phase1"], lr_phase2=p["lr_phase2"], weight_decay=p["weight_decay"],
            scheduler_factor=p["scheduler_factor"], scheduler_patience=p["scheduler_patience"],
            epochs_phase2=EPOCHS_PHASE2_SEARCH, type_loss_weight=type_loss_weight, trial=trial,
        )
        return val_f1
    finally:
        del model
        torch.cuda.empty_cache()


study_multitask = run_study("08_multitask", objective_multitask, N_TRIALS)

In [ ]:
# Sub-Step 6.4
# Tujuan: Studi 3/3 -- 09_noise_robust

def objective_noise_robust(trial):
    p = suggest_shared_params(trial)
    label_smoothing = trial.suggest_float("label_smoothing", 0.0, 0.2)
    mislabel_weight = trial.suggest_float("mislabel_weight", 0.1, 0.9)
    mistake_threshold = trial.suggest_float("mistake_threshold", 0.3, 0.7)

    flagged_mask = MISTAKE_SCORE > mistake_threshold
    sample_weights_fit = np.where(flagged_mask, mislabel_weight, 1.0)
    fit_loader, val_loader = make_loaders(BeanDataset, {"weights": sample_weights_fit.tolist()}, {}, p["batch_size"])
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing, reduction="none")
    model, head = build_model("efficientnet_b0", num_classes=4)
    try:
        _, val_f1 = train_one_model_hpo(
            model, head, fit_loader, val_loader, device, f"09_noise_robust_trial{trial.number}",
            lr_phase1=p["lr_phase1"], lr_phase2=p["lr_phase2"], weight_decay=p["weight_decay"],
            scheduler_factor=p["scheduler_factor"], scheduler_patience=p["scheduler_patience"],
            epochs_phase2=EPOCHS_PHASE2_SEARCH, criterion=criterion, trial=trial,
        )
        return val_f1
    finally:
        del model
        torch.cuda.empty_cache()


study_noise_robust = run_study("09_noise_robust", objective_noise_robust, N_TRIALS)

## Section 7 — Retrain Final dengan Hyperparameter Terbaik (Plafon Epoch Penuh)

Angka dari trial pencarian (Section 6) sengaja dipangkas plafon epoch-nya -- angka yang
dilaporkan sebagai hasil FINAL harus dari retrain dengan plafon penuh (45 epoch fase-2),
supaya adil dibandingkan dengan baseline `CBQD - Training.ipynb` (juga 45 epoch).

In [ ]:
# Sub-Step 7.1
# Tujuan: Retrain 05_convnext_tiny dengan best_params, evaluasi val+test

final_results = []


def _record_final(model_name, best_params, val_f1, test_metrics, baseline_test_f1):
    entry = {
        "model": model_name, "tuned_val_macro_f1": val_f1,
        "tuned_test_macro_f1": test_metrics["macro_f1"], "tuned_test_accuracy": test_metrics["accuracy"],
        "baseline_test_macro_f1": baseline_test_f1,
        "delta_test_macro_f1": test_metrics["macro_f1"] - baseline_test_f1,
    }
    for cls in CLASS_NAMES:
        entry[f"test_recall_{cls}"] = test_metrics["report"][cls]["recall"]
    entry.update({f"param_{k}": v for k, v in best_params.items()})
    final_results.append(entry)
    print(f"[{model_name}] FINAL tuned_val_f1={val_f1:.4f}  tuned_test_f1={test_metrics['macro_f1']:.4f}  "
          f"(baseline test_f1={baseline_test_f1:.4f}, delta={entry['delta_test_macro_f1']:+.4f})")
    return entry


bp = study_convnext.best_params
fit_loader, val_loader = make_loaders(BeanDataset, {}, {}, bp["batch_size"])
model, head = build_model("convnext_tiny", num_classes=4)
model, val_f1 = train_one_model_hpo(
    model, head, fit_loader, val_loader, device, "05_convnext_tiny_final",
    lr_phase1=bp["lr_phase1"], lr_phase2=bp["lr_phase2"], weight_decay=bp["weight_decay"],
    scheduler_factor=bp["scheduler_factor"], scheduler_patience=bp["scheduler_patience"],
    epochs_phase2=EPOCHS_PHASE2_FINAL,
)
test_metrics = evaluate(model, get_test_loader(bp["batch_size"]), device)
_record_final("05_convnext_tiny", bp, val_f1, test_metrics, baseline_test_f1=0.9479)
del model
torch.cuda.empty_cache()

In [ ]:
# Sub-Step 7.2
# Tujuan: Retrain 08_multitask dengan best_params, evaluasi val+test

bp = study_multitask.best_params
fit_loader, val_loader = make_loaders(MultiTaskDataset, {}, {}, bp["batch_size"])
mt_model = EfficientNetMultiTask()
mt_model, val_f1 = train_multitask_hpo(
    mt_model, fit_loader, val_loader, device, "08_multitask_final",
    lr_phase1=bp["lr_phase1"], lr_phase2=bp["lr_phase2"], weight_decay=bp["weight_decay"],
    scheduler_factor=bp["scheduler_factor"], scheduler_patience=bp["scheduler_patience"],
    epochs_phase2=EPOCHS_PHASE2_FINAL, type_loss_weight=bp["type_loss_weight"],
)


def multitask_predict_fn(images):
    out_damage, out_type = mt_model(images)
    damage_pred = out_damage.argmax(dim=1).cpu().numpy()
    type_pred = out_type.argmax(dim=1).cpu().numpy()
    return np.where(damage_pred == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT[t] for t in type_pred])


test_metrics = evaluate_combined(multitask_predict_fn, get_test_loader(bp["batch_size"]), device)
_record_final("08_multitask", bp, val_f1, test_metrics, baseline_test_f1=0.9397)
del mt_model
torch.cuda.empty_cache()

In [ ]:
# Sub-Step 7.3
# Tujuan: Retrain 09_noise_robust dengan best_params, evaluasi val+test

bp = study_noise_robust.best_params
flagged_mask = MISTAKE_SCORE > bp["mistake_threshold"]
sample_weights_fit = np.where(flagged_mask, bp["mislabel_weight"], 1.0)
fit_loader, val_loader = make_loaders(BeanDataset, {"weights": sample_weights_fit.tolist()}, {}, bp["batch_size"])
criterion = nn.CrossEntropyLoss(label_smoothing=bp["label_smoothing"], reduction="none")
model9, head9 = build_model("efficientnet_b0", num_classes=4)
model9, val_f1 = train_one_model_hpo(
    model9, head9, fit_loader, val_loader, device, "09_noise_robust_final",
    lr_phase1=bp["lr_phase1"], lr_phase2=bp["lr_phase2"], weight_decay=bp["weight_decay"],
    scheduler_factor=bp["scheduler_factor"], scheduler_patience=bp["scheduler_patience"],
    epochs_phase2=EPOCHS_PHASE2_FINAL, criterion=criterion,
)
test_metrics = evaluate(model9, get_test_loader(bp["batch_size"]), device)
_record_final("09_noise_robust", bp, val_f1, test_metrics, baseline_test_f1=0.9610)
del model9
torch.cuda.empty_cache()

## Section 8 — Ringkasan: Tuned vs Baseline

In [ ]:
# Sub-Step 8.1
# Tujuan: Simpan & cetak ringkasan akhir

Path("metadata").mkdir(exist_ok=True)
final_df = pd.DataFrame(final_results).sort_values("tuned_test_macro_f1", ascending=False).reset_index(drop=True)
final_df.to_csv("metadata/hpo_final_summary.csv", index=False)

print(f"DRY_RUN={DRY_RUN} -- angka di bawah ini {'BELUM final (epoch/trial kecil)' if DRY_RUN else 'hasil HPO penuh'}")
print()
pd.set_option("display.max_columns", None, "display.width", 250)
cols_show = ["model", "tuned_val_macro_f1", "tuned_test_macro_f1", "baseline_test_macro_f1", "delta_test_macro_f1"]
print("=== Tuned vs Baseline (diurutkan tuned_test_macro_f1, tertinggi dulu) ===")
print(final_df[cols_show].round(4).to_string(index=False))
print()
print("=== Best hyperparameters per model ===")
param_cols = [c for c in final_df.columns if c.startswith("param_")]
print(final_df[["model"] + param_cols].to_string(index=False))